idea: gpu/tpu/cpu optimized "numpy" via fusing operations.

JAX -> jaxpr -> xla -> StableHLO -> backend codegen (compilers) 

In [10]:
import jax, jax.numpy as jnp

def f(x):
    return jnp.sin(x) * 2 + 1
x = jnp.ones(3)
jax.jit(f)

<PjitFunction of <function f at 0x77bfa448b100>>

In [13]:
#jaxpr
jax.make_jaxpr(f)(x)

{ lambda ; a:f32[3]. let
    b:f32[3] = sin a
    c:f32[3] = mul b 2.0:f32[]
    d:f32[3] = add c 1.0:f32[]
  in (d,) }

Accelerated linear algebra (XLA)

In [19]:
# StableHLO pre XLA optimized
jax.jit(f).lower(x).as_text(dialect="hlo")

'HloModule jit_f, entry_computation_layout={(f32[3]{0})->f32[3]{0}}\n\nENTRY main.1 {\n  x.1 = f32[3]{0} parameter(0)\n  sin.1 = f32[3]{0} sine(x.1)\n  constant.3 = f32[] constant(2)\n  broadcast.3 = f32[3]{0} broadcast(constant.3), dimensions={}\n  mul.1 = f32[3]{0} multiply(sin.1, broadcast.3)\n  constant.2 = f32[] constant(1)\n  broadcast.2 = f32[3]{0} broadcast(constant.2), dimensions={}\n  ROOT add.1 = f32[3]{0} add(mul.1, broadcast.2)\n}\n\n'

In [ ]:
# HLO post fused kernel
jax.jit(f).lower(x).compile().as_text()

'HloModule jit_f, is_scheduled=true, entry_computation_layout={(f32[3]{0})->f32[3]{0}}, allow_spmd_sharding_propagation_to_parameters={true}, allow_spmd_sharding_propagation_to_output={true}\n\nFileNames\n1 "<frozen runpy>"\n2 "/tmp/ipykernel_638128/3120407226.py"\n3 "/tmp/ipykernel_638128/770840574.py"\n\nFunctionNames\n1 "_run_module_as_main"\n2 "_run_code"\n3 "<module>"\n4 "f"\n\nFileLocations\n1 {file_name_id=1 function_name_id=1 line=198 end_line=199 column=11 end_column=42}\n2 {file_name_id=1 function_name_id=2 line=88 end_line=88 column=4 end_column=27}\n3 {file_name_id=2 function_name_id=3 line=1 end_line=1 column=0 end_column=19}\n4 {file_name_id=3 function_name_id=4 line=4 end_line=4 column=11 end_column=29}\n\nStackFrames\n1 {file_location_id=1 parent_frame_id=1}\n2 {file_location_id=2 parent_frame_id=2}\n3 {file_location_id=3 parent_frame_id=3}\n4 {file_location_id=4 parent_frame_id=4}\n\n\n%fused_computation (param_0.2: f32[3]) -> f32[3] {\n  %param_0.2 = f32[3]{0} paramet

In [18]:
!XLA_FLAGS="--xla_dump_to=dump --xla_dump_fusion_visualization" python -c "import jax, jax.numpy as jnp; f = jax.jit(lambda x: jnp.sin(x) * 2 + 1); f(jnp.ones(1000)).block_until_ready()"

---

add autograd

In [35]:
def f(x):
    return jnp.sin(x) * 2 + 1
x = 2.0

lower = jax.jit(jax.grad(f)).lower(x)
lower

In [34]:
jax.make_jaxpr(jax.grad(f))(x)

{ lambda ; a:f32[]. let
    b:f32[] = sin a
    c:f32[] = cos a
    d:f32[] = mul b 2.0:f32[]
    _:f32[] = add d 1.0:f32[]
    e:f32[] = mul 1.0:f32[] 2.0:f32[]
    f:f32[] = mul e c
  in (f,) }

In [36]:
# StableHLO pre XLA optimized
lower.as_text(dialect="hlo")

'HloModule jit_f, entry_computation_layout={(f32[])->f32[]}\n\nENTRY main.1 {\n  x.1 = f32[] parameter(0)\n  cos.1 = f32[] cosine(x.1)\n  constant.1 = f32[] constant(2)\n  ROOT mul.1 = f32[] multiply(cos.1, constant.1)\n}\n\n'

In [37]:
# HLO post fused kernel
lower.compile().as_text()

'HloModule jit_f, is_scheduled=true, entry_computation_layout={(f32[])->f32[]}, allow_spmd_sharding_propagation_to_parameters={true}, allow_spmd_sharding_propagation_to_output={true}\n\nFileNames\n1 "<frozen runpy>"\n2 "/tmp/ipykernel_638128/2025706240.py"\n\nFunctionNames\n1 "_run_module_as_main"\n2 "_run_code"\n3 "<module>"\n4 "f"\n\nFileLocations\n1 {file_name_id=1 function_name_id=1 line=198 end_line=199 column=11 end_column=42}\n2 {file_name_id=1 function_name_id=2 line=88 end_line=88 column=4 end_column=27}\n3 {file_name_id=2 function_name_id=3 line=5 end_line=5 column=8 end_column=37}\n4 {file_name_id=2 function_name_id=4 line=2 end_line=2 column=11 end_column=25}\n\nStackFrames\n1 {file_location_id=1 parent_frame_id=1}\n2 {file_location_id=2 parent_frame_id=2}\n3 {file_location_id=3 parent_frame_id=3}\n4 {file_location_id=4 parent_frame_id=4}\n\n\n%fused_computation (param_0.2: f32[]) -> f32[] {\n  %param_0.2 = f32[] parameter(0)\n  %cos.0 = f32[] cosine(%param_0.2), metadata={

---

In [39]:
compiled = lower.compile()
compiled.cost_analysis()

{'flops': 1.0,
 'transcendentals': 1.0,
 'bytes accessed': 8.0,
 'bytes accessed0{}': 4.0,
 'utilization0{}': 1.0,
 'bytes accessed1{}': 4.0,
 'utilization1{}': 1.0,
 'bytes accessedout{}': 4.0}

In [41]:
m = compiled.memory_analysis()
print(m.argument_size_in_bytes)        # inputs
print(m.output_size_in_bytes)          # outputs
print(m.temp_size_in_bytes)            # scratch/intermediate buffers
print(m.alias_size_in_bytes)           # memory shared between inputs and outputs (donation)
print(m.generated_code_size_in_bytes)

4
4
0
0
0


---

In [45]:
from jax import export

def f(x):
    return jnp.sin(x) * 2 + 1

# 1. Trace + lower for a given input shape/dtype
exp = export.export(jax.jit(f))(jax.ShapeDtypeStruct((1000,), jnp.float32))

# 2. Serialize to bytes (save to disk, ship anywhere)
blob = exp.serialize()
blob

bytearray(b'<\x00\x00\x008\x00`\x00^\x00X\x00T\x00P\x00L\x00H\x00\x00\x00@\x00<\x008\x004\x000\x00,\x00(\x00&\x00 \x00\x00\x00\x00\x00D\x00\x1c\x00\x18\x00\x14\x00\x10\x00\x0c\x00\x08\x00\x04\x008\x00\x00\x00\\\x00\x00\x00`\x00\x00\x00d\x00\x00\x00h\x00\x00\x00l\x00\x00\x00l\x00\x00\x00l\x00\x00\x00\x90\x00\x00\x00\x00\x00\n\x00\x90\x00\x00\x00\xc4\x03\x00\x00\xc4\x03\x00\x00\xc4\x03\x00\x00\xa8\x03\x00\x00\xc0\x03\x00\x00\xc8\x03\x00\x00\x01\x00\x00\x00\xcc\x03\x00\x00\xf0\x03\x00\x00\xf0\x03\x00\x00 \x04\x00\x00t\x04\x00\x00\x00\x00\x03\x00\x01\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\x04\x00\x00\x00H\xfc\xff\xff\x00\x00\x01\n\x04\x00\x00\x00\x01\x00\x00\x00\x04\x00\x00\x00\x04\x00\x00\x001000\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00"\x03\x00\x00ML\xefR\rStableHLO_v1.16.1\x00\x01!\x05\x01\x05\x11\x01\x03\x0b\x03\x0f\x0f\x13\x17\x1b\x1f#\'\x03\x93o\

In [46]:
# 3. Later, even in another process without f defined
exp2 = export.deserialize(blob)
exp2.call(jnp.zeros(1000))

Array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1.